# HW6: 판매 총액 예측 모델 선택과 Flask API
김준혁 · 22011725

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import platform
from sklearn.model_selection import train_test_split, cross_val_score

fonts = {"Windows": "Malgun Gothic", "Darwin": "AppleGothic"}
plt.rcParams["font.family"] = fonts.get(platform.system(), "NanumGothic")
plt.rcParams["axes.unicode_minus"] = False

## ① 읽고 나눈다 : 총액을 수량과 단가로

In [2]:
sales = pd.read_csv("sales_clean.csv")
X = sales[["quantity", "unit_price"]]
y = sales["total_sales"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)
print("training :", len(X_train), " test :", len(X_test))

training : 803  test : 201


## ② 선형 회귀와 결정 트리

In [3]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

lin = LinearRegression().fit(X_train, y_train)
tree = DecisionTreeRegressor(random_state=0).fit(X_train, y_train)
print("linear regression  test R² :", round(lin.score(X_test, y_test), 3))
print("decision tree      test R² :", round(tree.score(X_test, y_test), 3))

linear regression  test R² : 0.862
decision tree      test R² : 0.994


## ③ 특징 하나를 더한다 : 수량 × 단가

In [4]:
X2_train = X_train.assign(qp=X_train["quantity"] * X_train["unit_price"])
X2_test  = X_test.assign(qp=X_test["quantity"] * X_test["unit_price"])
lin2 = LinearRegression().fit(X2_train, y_train)
print("linear + qp        test R² :", round(lin2.score(X2_test, y_test), 3))
print("coef :", lin2.coef_.round(3), " intercept :", round(lin2.intercept_, 1))

linear + qp        test R² : 1.0
coef : [0. 0. 1.]  intercept : -0.0


	수량과 단가만 넣은 선형회귀의 테스트 R²는 0.862였다. 총액은 수량과 단가의 곱이므로 qp = quantity × unit_price를 입력에 추가하자 qp의 계수가 1이 되고 테스트 R²가 1.0이 됐다.

## ④ max_depth 를 바꾸면 : training 과 test

In [5]:
rows = []
for dep in [1, 2, 3, 5, 8, 12, None]:
    m = DecisionTreeRegressor(max_depth=dep, random_state=0).fit(X_train, y_train)
    rows.append([str(dep), m.get_n_leaves(),
                 round(m.score(X_train, y_train), 3), round(m.score(X_test, y_test), 3)])
pd.DataFrame(rows, columns=["max_depth", "leaves", "training R²", "test R²"])

,max_depth,leaves,training R²,test R²
0,1,2,0.361,0.346
1,2,4,0.739,0.711
2,3,8,0.876,0.846
3,5,32,0.972,0.961
4,8,250,0.998,0.991
5,12,781,1.000,0.994
6,None,800,1.000,0.994


## ⑤ 교차검증 : 나눔 한 번을 믿지 않는다

In [6]:
for dep in [3, 5, 8, 12, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=dep, random_state=0), X, y, cv=5)
    print(f"max_depth {str(dep):4s}  mean {s.mean():.3f}  std {s.std():.3f}")

max_depth 3     mean 0.845  std 0.019
max_depth 5     mean 0.961  std 0.003
max_depth 8     mean 0.992  std 0.000
max_depth 12    mean 0.995  std 0.000
max_depth None  mean 0.995  std 0.001


	비교용 결정 트리는 깊이 8을 선택했다. 교차검증 평균 R²는 깊이 8에서 0.992, 깊이 12에서 0.995로 차이가 0.003이지만, 잎 수는 250개와 781개여서 깊이 8이 더 단순하다.

## ⑥ 주문 한 건의 총액 예측

In [7]:
order = pd.DataFrame([[40, 200000]], columns=["quantity", "unit_price"])
order2 = order.assign(qp=order["quantity"] * order["unit_price"])

tree8 = DecisionTreeRegressor(max_depth=8, random_state=0).fit(X_train, y_train)

print("선형회귀 + qp :", int(round(lin2.predict(order2)[0])))
print("결정 트리(깊이 8) :", int(tree8.predict(order)[0]))
print("직접 계산 :", 40 * 200000)

선형회귀 + qp : 8000000
결정 트리(깊이 8) : 7362301
직접 계산 : 8000000


## ⑦ API에 사용할 모델과 이유

API에는 선형회귀 + qp를 사용한다. 수량 40개, 단가 200,000원인 주문에서 이 모델은 실제 총액과 같은 8,000,000원을 예측했고, 깊이 8의 결정 트리는 7,362,301원을 예측했다. 총액 공식이 정해진 실제 서비스라면 모델 없이 곱셈으로도 계산할 수 있다.

In [8]:
import joblib

joblib.dump(lin2, "sales_model.pkl")

['sales_model.pkl']